# Download Data

## 1. Config

In [1]:
import sys
from pathlib import Path
import pandas as pd
from datetime import datetime
from random import random
from time import time
import shutil

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import os
import tarfile
import requests
from tqdm import tqdm
from dotenv import load_dotenv
from src.download.downloader import (
    get_missing_data_list,
    download_data_archive,
    extract_data_archive
)

In [ ]:
load_dotenv()

BASE_URL = os.getenv("DAIC_WOZ_URL")
EXTENDED_BASE_URL = os.getenv("EXTENDED_DAIC_WOZ_URL")
DATA_START_ID = 600
DATA_END_ID = 718

RAW_DIR = PROJECT_ROOT / "data" / "raw"
AUDIO_DIR = PROJECT_ROOT / "data" / "audio"
TRANSCRIPT_DIR = PROJECT_ROOT / "data" / "transcript"

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

RAW_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)


LOG_DIR = PROJECT_ROOT / "logs"
SUCCESS_LOG = LOG_DIR / "download_success.csv"
FAILED_LOG = LOG_DIR / "download_failures.csv"


assert BASE_URL is not None, "DAIC_WOZ_URL not found in .env"
assert EXTENDED_BASE_URL is not None, "EXTENDED_DAIC_WOZ_URL not found in .env"

## 2. Download Audio and Transcripts

In [3]:
miss_data_list = get_missing_data_list(
    start_id=DATA_START_ID,
    end_id=DATA_END_ID,
    audio_dir=AUDIO_DIR,
    transcript_dir=TRANSCRIPT_DIR
)
print(f"Missing data IDs: {miss_data_list}")

Completed : 0
Corrupted : 0
Missing   : 193
Need Download : 193
Missing data IDs: [300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 48

In [4]:
success_count = 0
failed_count = 0

for data_id in tqdm(
    miss_data_list,
    desc="Downloading DAIC-WOZ",
    unit="data",
):
    archive_path = None

    try:
        download_result = download_data_archive(
            data_id=data_id,
            base_url=EXTENDED_BASE_URL,
            output_dir=RAW_DIR,
        )

        archive_path = download_result["file_path"]

        extract_result = extract_data_archive(
            archive_path=archive_path,
            audio_dir=AUDIO_DIR,
            transcript_dir=TRANSCRIPT_DIR,
        )

        if archive_path.exists():
            archive_path.unlink()

        success_row = pd.DataFrame([
            {
                "data_id": data_id,
                "audio_path": str(extract_result["audio_path"]),
                "transcript_path": str(extract_result["transcript_path"]),
                "timestamp": datetime.now().isoformat(),
            }
        ])

        success_row.to_csv(
            SUCCESS_LOG,
            mode="a",
            header=not SUCCESS_LOG.exists(),
            index=False,
        )

        success_count += 1
        tqdm.write(f"[SUCCESS] {data_id}")

    except Exception as e:
        if archive_path is not None and archive_path.exists():
            archive_path.unlink()

        tmp_path = RAW_DIR / f"{data_id}_P.tar.gz.tmp"
        if tmp_path.exists():
            tmp_path.unlink()

        failed_row = pd.DataFrame([
            {
                "data_id": data_id,
                "error_type": type(e).__name__,
                "error": str(e),
                "timestamp": datetime.now().isoformat(),
            }
        ])

        failed_row.to_csv(
            FAILED_LOG,
            mode="a",
            header=not FAILED_LOG.exists(),
            index=False,
        )

        failed_count += 1
        tqdm.write(f"[FAILED] {data_id} | {type(e).__name__}: {e}")
    

print("\n====================")
print(f"Success : {success_count}")
print(f"Failed  : {failed_count}")
print("====================")

[DOWNLOAD] 300_P.tar.gz
[DONE] 300_P.tar.gz (266.85 MB)


[SUCCESS] 300
[DOWNLOAD] 301_P.tar.gz
[DONE] 301_P.tar.gz (357.80 MB)


[SUCCESS] 301
[DOWNLOAD] 302_P.tar.gz
[DONE] 302_P.tar.gz (292.29 MB)


[SUCCESS] 302
[DOWNLOAD] 303_P.tar.gz
[DONE] 303_P.tar.gz (391.46 MB)


[SUCCESS] 303
[DOWNLOAD] 304_P.tar.gz
[DONE] 304_P.tar.gz (337.78 MB)


...

[SUCCESS] 488
[DOWNLOAD] 489_P.tar.gz
[DONE] 489_P.tar.gz (300.68 MB)


[SUCCESS] 489
[DOWNLOAD] 490_P.tar.gz
[DONE] 490_P.tar.gz (260.53 MB)


[SUCCESS] 490
[DOWNLOAD] 491_P.tar.gz
[DONE] 491_P.tar.gz (328.27 MB)


[SUCCESS] 491
[DOWNLOAD] 492_P.tar.gz
[DONE] 492_P.tar.gz (373.05 MB)


[SUCCESS] 492

Success : 189
Failed  : 4
